In [109]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [110]:
import sys
import numpy as np
import polars as pl  
from pathlib import Path
from sklearn.metrics import confusion_matrix,accuracy_score, roc_auc_score

sys.path.append('../')
from NeuroHDC.FHRR import *
from NeuroHDC.fn import *
from conformalHDC.models import *
from conformalHDC.methods import *
from conformalHDC.utils import *

In [111]:
# NOTE: When split the train, there might be a chance some neurons not firing at all. 

# -----------------------------
# load data
# ----------------------------- 
# NOTE. scale params, control the width for guassian kernel
# small value makes the code robust to small rate changes
# big value make class more separate
beta = 0.3
dim = 15000 # 10k-20k
training_window = (200,600)
my_bin_size = 25
iseed = 0
# training_window = (250,500)
# my_bin_size = 25 

in_path = Path("../data/rat") / f"odor_prep_{training_window}_{my_bin_size}.pickle"

cf_list = []
cf_list_il = []

# rat_name = ['Barat','Buchanan','Mitt','Stella','Superchris']
for irat in range(0,5):
    splits= prep_loader(
        irat = irat, 
        split_ratio= (0.5,0.4,0.1), # train, calibration, test,
        in_path=in_path,
        seed = iseed
        )

    Xtr_z, Xva_z, Xte_z = splits.train.X, splits.val.X, splits.test.X

    # print("train", summary(splits.train))
    # print("val", summary(splits.val))
    # print("test", summary(splits.test))

    # -----------------------------
    # Step 2.1: Build feature Encoder
    # -----------------------------
    n,nT,p = Xtr_z.shape # num of features

    rff = RFF(n_feature=p, dimension=dim, seed = iseed)

    # generate basis with MVN
    cov = np.eye(p) # NOTE maybe try sample correlation matrix?

    W = rff.gen_basis(cov = cov) # (dim, p)
    TB = rff.gen_time_base()

    rff.W = W
    rff.TB = TB
    
    # encode training
    encoded_x_train = rff.encode_all(
                        W_neurons=W, 
                        X = Xtr_z, 
                        Time_Base=TB,
                        beta = beta)

    # encode test and val data 
    encoded_x_val = rff.encode_all(W,Xva_z, 
                        Time_Base=TB,
                        beta = beta)
    encoded_x_te = rff.encode_all(W,Xte_z, 
                        Time_Base=TB,
                        beta = beta)

    # print(B.shape, W.shape, encoded_x_train.shape)
    # -----------------------------
    # Step 2.2: build prototype
    # ----------------------------- 

    class_proto = rff.build_class_prototypes(
        encoded_x_train,splits.train.y 
        ) 
    
    # #NOTE Optional: Use Iterative Novelty Learning
    # only 1-2 rats get better
    # class_proto_noval = rff.train_iterative_novelty(
    #     encoded_x_train,splits.train.y, iterations=5
    #     ) 
    
    # -----------------------------
    # Step 2.3: evaluate one-pass performance
    # -----------------------------
    y_tr_pred = rff.decode(encoded_x_train,class_proto)

    y_val_pred = rff.decode(encoded_x_val,class_proto)
    y_te_pred = rff.decode(encoded_x_te,class_proto)

    label = np.unique(splits.train.y)

    tr_acc = np.mean(y_tr_pred == splits.train.y)
    val_acc = np.mean(y_val_pred == splits.val.y)
    te_acc = np.mean(y_te_pred == splits.test.y)


    cf_v = confusion_matrix(splits.val.y, y_val_pred)
    cf_t = confusion_matrix(splits.test.y, y_te_pred)
    cf_list.append(cf_v+cf_t)

    print(
        f"ACC for rat {irat}:\n"
        f"  train: {accuracy_score(y_tr_pred,splits.train.y):.3f} \n"
        f"  val:   {accuracy_score(y_val_pred,splits.val.y):.3f}\n"
        f"  test:   {accuracy_score(y_te_pred,splits.test.y):.3f}\n"
        f"  chance:  {1/len(label):.3f}"
        # f" confusion matrix:\n {cf}"
    ) 

ACC for rat 0:
  train: 0.955 
  val:   0.481
  test:   0.385
  chance:  0.250
ACC for rat 1:
  train: 0.989 
  val:   0.643
  test:   0.647
  chance:  0.250
ACC for rat 2:
  train: 1.000 
  val:   0.573
  test:   0.619
  chance:  0.250
ACC for rat 3:
  train: 1.000 
  val:   0.705
  test:   0.933
  chance:  0.250
ACC for rat 4:
  train: 1.000 
  val:   0.833
  test:   0.875
  chance:  0.250


In [94]:
encoded_x_train = rff.encode_all(
                        W_neurons=W, 
                        X = Xtr_z, 
                        Time_Base=TB,
                        beta = beta)
print(encoded_x_train.shape)

(82, 15000)


In [95]:
Xtr_z.shape

(82, 16, 46)

In [148]:
irat = 2
ID_CLASSES = [0,1,2,3]
splits = prep_loader(
    irat=irat, 
    split_ratio=(0.5, 0.4, 0.1), 
    in_path=in_path, 
    seed=iseed
)
X_tr_id, y_tr_id = splits.train.X, splits.train.y
X_cal_id, y_cal_id = splits.val.X, splits.val.y
X_te_id, y_te_id = splits.test.X, splits.test.y

In [149]:
print(f"number of training points: {len(splits.train.y)}.")
print(f"number of calibration points: {len(splits.val.y)}.")
print(f"number of test points: {len(splits.test.y)}.")

number of training points: 104.
number of calibration points: 82.
number of test points: 21.


In [150]:
splits.train.X.shape

(104, 16, 104)

In [151]:
splits.train.y.shape

(104,)

In [152]:
run_window = (0,200)
run_in_path = Path("../data/rat") / f"run_prep_{run_window}_{my_bin_size}.pickle"

with run_in_path.open("rb") as f:
    run_data = pickle.load(f)

# NO normalization needed, just load raw
X_ood = run_data[irat]['binned_spk'] 

print(f"OOD Data shape (Running): {X_ood.shape}")

OOD Data shape (Running): (23, 8, 104)


In [153]:
rff = RFF(n_feature=X_tr_id.shape[2], dimension=dim, seed=iseed)
W = rff.gen_basis(cov=np.eye(X_tr_id.shape[2]))
TB = rff.gen_time_base()

# Encode everything
print("Encoding data...")
enc_tr_id = rff.encode_all(W, X_tr_id, TB, beta)
enc_cal_id = rff.encode_all(W, X_cal_id, TB, beta)
enc_te_id = rff.encode_all(W, X_te_id, TB, beta)
enc_ood = rff.encode_all(W, X_ood, TB, beta)

# Build Prototypes using RFF
proto_dict = rff.build_class_prototypes(enc_tr_id, y_tr_id)
proto_matrix = np.stack([proto_dict[k] for k in ID_CLASSES])

Encoding data...


In [154]:
enc_tr_id.shape

(104, 15000)

In [155]:
SCORE_TYPES = ['sim', 'ratio', 'discount',"penalized", "inverse_quantile"] 
ALPHA = 0.1  # Fixed alpha for 90% target coverage
results_list = []

In [156]:
chdc = ConformalHDC(
    class_HVs=proto_matrix, 
    class_labels=ID_CLASSES, 
    sim_measure="complex_cosine"
)

In [157]:
preds_vanilla_idx = chdc.predict(enc_te_id)
preds_vanilla = np.array([ID_CLASSES[i] for i in preds_vanilla_idx])
baseline_acc = accuracy_score(y_te_id, preds_vanilla)
baseline_acc

0.6190476190476191

In [158]:
for score_type in SCORE_TYPES:
    print(f"Testing Score Type: {score_type}...")
    
    # A. Calibrate using the specific score type
    # This updates self.scores_calib inside the object
    chdc.compute_calib_scores(enc_cal_id, y_cal_id, score_type=score_type)
    
    # Task 1: Set-Valued Prediction (Coverage & Efficiency)
    sets = chdc.set_valued_CP(enc_te_id, ALPHA, marginal=True)
    
    # Evaluate sets (Coverage, Avg Size)
    # df_res contains 'M-coverage', 'M-size', etc.
    df_res = eval_m_psets(sets, y_te_id)
    marginal_cov = df_res['M-coverage'].item()
    avg_size = df_res['M-size'].item()

    # Task 2: point-valued conformalHDC
    preds = chdc.point_valued_CP(enc_te_id, method="efficient")
    point_acc = accuracy_score(y_te_id, preds)
    
    # Task 3: OOD Detection (AUROC)
    # Get p-values for ID and OOD samples
    p_vals_id  = chdc.get_max_p_value(enc_te_id, marginal=True)
    p_vals_ood = chdc.get_max_p_value(enc_ood, marginal=True)
    
    # AUROC Calculation
    y_true_roc = np.concatenate([np.ones(len(p_vals_id)), np.zeros(len(p_vals_ood))])
    y_scores_roc = np.concatenate([p_vals_id, p_vals_ood])
    ood_auroc = roc_auc_score(y_true_roc, y_scores_roc)
    
    # Record Results
    results_list.append({
        "Method": score_type,
        "Alpha": ALPHA,
        "Baseline_Acc": baseline_acc,
        "Point_Acc": point_acc,
        "Set_Coverage": marginal_cov,
        "Set_Size": avg_size,
        "OOD_AUROC": ood_auroc
    })

# ---------------------------------------------------------
# 5. RESULTS TABLE
# ---------------------------------------------------------
final_df = pd.DataFrame(results_list)
print("\nFinal Comparison Results:")
print(final_df.round(4).to_string(index=False))

Testing Score Type: sim...
Testing Score Type: ratio...
Testing Score Type: discount...
Testing Score Type: penalized...
Testing Score Type: inverse_quantile...

Final Comparison Results:
          Method  Alpha  Baseline_Acc  Point_Acc  Set_Coverage  Set_Size  OOD_AUROC
             sim    0.1         0.619      0.619        0.9048    3.6190     1.0000
           ratio    0.1         0.619      0.619        0.8095    2.6190     0.5383
        discount    0.1         0.619      0.619        0.9048    3.5238     1.0000
       penalized    0.1         0.619      0.619        0.9524    3.7619     0.0000
inverse_quantile    0.1         0.619      0.619        0.8095    2.7143     0.4865


In [57]:
len(y_tr_id)+len(y_cal_id)+len(y_te_id)

123

In [59]:
len(X_ood)

29